# La Paz Southern Sky Survey — Emission-Line Object Pipeline

**Paper:** Zafar T.A. & Hudec R. (2025) — *Feasibility study for novel application of Southern Sky La Paz survey low dispersive spectral plates in recent astrophysics*, Astronomische Nachrichten.

**GitHub:** https://github.com/YOUR-USERNAME/lapaz-survey-pipeline

---

## Pipeline Overview

This notebook implements a two-stage deep learning pipeline to extract emission-line object candidates from digitized archival low-dispersion spectroscopy (LDS) photographic plates of the La Paz Southern Sky Survey (Bolivia, 1926–1929).

| Stage | Cell | Method |
|-------|------|--------|
| Setup | Cell 1 | Install packages, configure paths |
| Upload | Cell 2 | Upload digitized plate images |
| Detection + Anomaly Scoring | Cell 3 | Background subtraction → connected components → 1D autoencoder → Isolation Forest |
| Profile Visualization | Cell 3b | Plot low-score vs high-score spectral profiles |
| Human Review | Cell 4 | Interactive approve/reject widget → binary masks |
| Supervised Classification | Cell 5 | Fine-tuned ResNet18 on 64×64 cutouts → CSV catalogue |

**Hardware:** Google Colab with T4 GPU recommended.  
**Runtime:** < 2 hours for 168 plates on a single T4 GPU.


## Cell 1 — Install Packages & Setup

Installs required libraries and creates working directories under `/content/lapaz/`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install packages & setup
# ═══════════════════════════════════════════════════════════════════════════════

!pip install -q segmentation-models-pytorch albumentations scikit-learn opencv-python-headless

import cv2, numpy as np, time, random, zipfile, os, csv
from pathlib import Path
from sklearn.ensemble import IsolationForest
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from ipywidgets import (VBox, HBox, Output, Button, Label,
                        FloatSlider, IntSlider, Dropdown, HTML, Layout)
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab import files

# ── Working directories ───────────────────────────────────────────────────────
WORK       = Path('/content/lapaz')
PLATES_DIR = WORK / 'plates'
MASKS_DIR  = WORK / 'masks'
CKPT_DIR   = WORK / 'checkpoints'
for d in [PLATES_DIR, MASKS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PLATE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('✅ Ready — run Cell 2 to upload your plates')


## Cell 2 — Upload Plate Images

Upload your digitized plate JPEG/PNG/TIFF files. Files are saved to `/content/lapaz/plates/`.

> **Tip:** Select all plates at once using Ctrl+Click (Windows) or Cmd+Click (Mac).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Upload plate images
# ═══════════════════════════════════════════════════════════════════════════════

print('Select ALL your plate images (jpg/png/tif).')
print('Hold Ctrl (Windows) or Cmd (Mac) to select multiple files at once.')
print()

uploaded = files.upload()

saved = []
for fname, data in uploaded.items():
    ext = Path(fname).suffix.lower()
    if ext in PLATE_EXTS:
        dest = PLATES_DIR / fname
        dest.write_bytes(data)
        saved.append(dest)
        print(f'  ✓  {fname}  ({len(data)/1024:.0f} KB)')
    else:
        print(f'  ✗  {fname}  skipped (not a recognised image format)')

all_plates = sorted(saved)
print(f'\n{len(all_plates)} plate(s) uploaded and ready')
print('Run Cell 3 to detect spectra and train the anomaly detector')


## Cell 3 — Spectral Detection + Autoencoder Anomaly Scoring

**Part A — Detection:** Multi-scale median background subtraction, thresholding, morphological filtering, and connected component analysis extract spectral candidate positions from each plate.

**Part B — Autoencoder:** A 1D convolutional autoencoder (encoder: Conv1d→BN→ReLU→MaxPool×2→Linear; decoder: mirror) is trained unsupervised on all extracted 64-pixel spectral profiles.

**Part C — Scoring:** Combined anomaly score = 0.6 × reconstruction error + 0.4 × Isolation Forest score. Top 5% (95th percentile) are forwarded for review.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Detect spectra + train autoencoder + score everything
# ═══════════════════════════════════════════════════════════════════════════════

SPEC_LEN = 64
HALF_W   = 3

# ─────────────────────────────────────────────────────────────────────────────
# Part A: Spectrum detection and 1D extraction
# ─────────────────────────────────────────────────────────────────────────────

def detect_and_extract(plate_path):
    img = cv2.imread(str(plate_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f'  [ERROR] Cannot read {plate_path}')
        return []
    h, w = img.shape

    records = []
    for ksize in [51, 31, 71]:
        if ksize >= min(h, w):
            continue
        bg         = cv2.medianBlur(img, ksize)
        diff_light = np.clip(bg.astype(np.int16) - img.astype(np.int16), 0, 255).astype(np.uint8)
        diff_dark  = np.clip(img.astype(np.int16) - bg.astype(np.int16), 0, 255).astype(np.uint8)
        diff       = diff_light if diff_light.mean() >= diff_dark.mean() else diff_dark

        nonzero = diff[diff > 0]
        if len(nonzero) == 0:
            continue
        thresh_val = max(5, int(np.percentile(nonzero, 20)))
        _, thresh  = cv2.threshold(diff, thresh_val, 255, cv2.THRESH_BINARY)

        k_h   = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 1))
        k_e   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_DILATE, k_h, iterations=2)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_ERODE,  k_e, iterations=1)

        n, _, stats, centroids = cv2.connectedComponentsWithStats(thresh, 8)
        margin = 8

        for i in range(1, n):
            area = stats[i, cv2.CC_STAT_AREA]
            if not (8 <= area <= 20000):
                continue
            x  = stats[i, cv2.CC_STAT_LEFT]
            y  = stats[i, cv2.CC_STAT_TOP]
            bw = stats[i, cv2.CC_STAT_WIDTH]
            bh = stats[i, cv2.CC_STAT_HEIGHT]
            cx = int(centroids[i][0])
            cy = int(centroids[i][1])

            if (x < margin or y < margin or
                    x + bw > w - margin or y + bh > h - margin):
                continue

            x0, x1 = cx - SPEC_LEN // 2, cx + SPEC_LEN // 2
            y0, y1 = cy - HALF_W, cy + HALF_W + 1
            if x0 < 0 or x1 > w or y0 < 0 or y1 > h:
                continue

            strip   = img[y0:y1, x0:x1].astype(np.float32)
            nrows   = strip.shape[0]
            weights = np.array([nrows // 2 - abs(j - nrows // 2) + 1
                                 for j in range(nrows)], dtype=np.float32)
            profile = (strip * weights[:, None]).sum(0) / weights.sum()
            mu, sigma = profile.mean(), profile.std()
            if sigma < 1e-4:
                continue
            profile = ((profile - mu) / sigma).astype(np.float32)

            records.append(dict(
                plate=plate_path.name, cx=cx, cy=cy,
                area=int(area), aspect=float(bw / max(bh, 1)),
                profile=profile))

        if len(records) > 100:
            break

    records.sort(key=lambda r: r['area'], reverse=True)
    kept, seen = [], []
    for r in records:
        if all(abs(r['cx'] - s[0]) > 10 or abs(r['cy'] - s[1]) > 10
               for s in seen):
            kept.append(r)
            seen.append((r['cx'], r['cy']))
    return kept


print('Extracting spectra from all plates ...')
print()
all_records = []

# ── Save Figure 7 pipeline illustration ──────────────────────────────────────
plate_path_fig = sorted(PLATES_DIR.glob('*.jpg'))[0]
img_fig = cv2.imread(str(plate_path_fig), cv2.IMREAD_GRAYSCALE)
crop_r  = 200; crop_c = 200; crop_h = 400; crop_w = 400

crop      = img_fig[crop_r:crop_r+crop_h, crop_c:crop_c+crop_w]
bg_fig    = cv2.medianBlur(img_fig, 51)
diff_fig  = cv2.absdiff(bg_fig, img_fig)
diff_crop = diff_fig[crop_r:crop_r+crop_h, crop_c:crop_c+crop_w]

nz = diff_fig[diff_fig > 0]
tau = int(max(5, np.percentile(nz, 20))) if len(nz) else 5
_, thresh_fig = cv2.threshold(diff_fig, tau, 255, cv2.THRESH_BINARY)
k_h = cv2.getStructuringElement(cv2.MORPH_RECT,    (9, 1))
k_e = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
thresh_fig = cv2.dilate(thresh_fig, k_h, iterations=2)
thresh_fig = cv2.erode (thresh_fig, k_e, iterations=1)
mask_crop  = thresh_fig[crop_r:crop_r+crop_h, crop_c:crop_c+crop_w]

os.makedirs('/content/lapaz/figures', exist_ok=True)
cv2.imwrite('/content/lapaz/figures/step1_raw.png',  crop)
cv2.imwrite('/content/lapaz/figures/step2_diff.png', diff_crop)
cv2.imwrite('/content/lapaz/figures/step3_mask.png', mask_crop)
print('Pipeline illustration panels saved.')

# ── Main detection loop ───────────────────────────────────────────────────────
for plate_path in all_plates:
    recs = detect_and_extract(plate_path)
    all_records.extend(recs)
    print(f'  {plate_path.name:45s}  {len(recs):6,d} spectra')
print(f'\nTotal spectra extracted: {len(all_records):,}')

if all_plates and all_records:
    img0  = cv2.imread(str(all_plates[0]), cv2.IMREAD_GRAYSCALE)
    vis   = cv2.cvtColor(img0, cv2.COLOR_GRAY2RGB)
    first = all_plates[0].name
    for r in [x for x in all_records if x['plate'] == first][:300]:
        cv2.circle(vis, (r['cx'], r['cy']), 8, (255, 50, 50), 2)
    scale = min(1.0, 900 / max(img0.shape))
    disp  = cv2.resize(vis, (int(img0.shape[1]*scale), int(img0.shape[0]*scale)))
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(disp)
    ax.set_title(f'Detected spectra on {first}  '
                 f'(red circles = {len([x for x in all_records if x["plate"]==first])} spectra)',
                 fontsize=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if len(all_records) == 0:
    raise RuntimeError(
        'No spectra detected!\n'
        'Please check:\n'
        '  * Did your plates upload correctly in Cell 2?\n'
        '  * Are the images full-resolution scans (not thumbnails)?\n'
        '  * Upload at least 5-10 plates for best results.')


# ─────────────────────────────────────────────────────────────────────────────
# Part B: 1D Convolutional Autoencoder
# ─────────────────────────────────────────────────────────────────────────────

class SpectralAE(nn.Module):
    def __init__(self, seq_len=64, latent=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear(64 * (seq_len // 4), latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 64 * (seq_len // 4)), nn.ReLU(),
            nn.Unflatten(1, (64, seq_len // 4)),
            nn.Upsample(scale_factor=2),
            nn.Conv1d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv1d(64, 32, 5, padding=2), nn.ReLU(),
            nn.Conv1d(32, 1, 5, padding=2),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


profiles = np.stack([r['profile'] for r in all_records])
X        = torch.tensor(profiles[:, None, :])

BATCH  = min(512, max(4, len(all_records)))
loader = DataLoader(TensorDataset(X), batch_size=BATCH,
                    shuffle=True, drop_last=False, num_workers=0)

AE_CKPT = CKPT_DIR / 'autoencoder.pth'
ae      = SpectralAE(seq_len=SPEC_LEN, latent=16).to(DEVICE)
opt     = optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-5)
n_epoch = 80
start   = 0

if AE_CKPT.exists():
    ck = torch.load(AE_CKPT, map_location=DEVICE)
    ae.load_state_dict(ck['model'])
    start = ck['epoch'] + 1
    print(f'Resumed autoencoder from epoch {start}')

sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epoch - start)

print(f'\nTraining autoencoder on {len(all_records):,} spectra '
      f'(batch={BATCH}, epochs={n_epoch}) ...')
print(f'{"Epoch":>6} | {"Loss":>10}')
print('-' * 22)

losses = []
for ep in range(start, n_epoch):
    ae.train()
    total = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        recon, _ = ae(xb)
        loss = F.mse_loss(recon, xb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item()
    sched.step()
    losses.append(total / len(loader))
    if (ep + 1) % 10 == 0 or ep == n_epoch - 1:
        print(f'{ep+1:>6} | {losses[-1]:>10.6f}')
    torch.save({'epoch': ep, 'model': ae.state_dict()}, AE_CKPT)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='steelblue')
ax.set(title='Autoencoder training loss', xlabel='Epoch', ylabel='MSE')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Autoencoder trained and saved')


# ─────────────────────────────────────────────────────────────────────────────
# Part C: Score every spectrum
# ─────────────────────────────────────────────────────────────────────────────

print('\nScoring all spectra ...')
ae.eval()
recon_errs, latents = [], []

with torch.no_grad():
    for i in range(0, len(X), 1024):
        xb = X[i:i+1024].to(DEVICE)
        recon, z = ae(xb)
        recon_errs.append(((recon - xb) ** 2).mean(dim=[1, 2]).cpu().numpy())
        latents.append(z.cpu().numpy())

recon_errs = np.concatenate(recon_errs)
latents    = np.concatenate(latents)

contamination = min(0.1, max(0.01, 50 / max(len(all_records), 1)))
iso       = IsolationForest(contamination=contamination, n_estimators=200,
                             random_state=42, n_jobs=-1)
iso_score = -iso.fit(latents).score_samples(latents)

def _norm(v):
    mn, mx = v.min(), v.max()
    return (v - mn) / max(mx - mn, 1e-9)

combined = 0.6 * _norm(recon_errs) + 0.4 * _norm(iso_score)

for i, r in enumerate(all_records):
    r['recon_err']     = float(recon_errs[i])
    r['iso_score']     = float(iso_score[i])
    r['anomaly_score'] = float(combined[i])

all_records.sort(key=lambda r: r['anomaly_score'], reverse=True)
scores = np.array([r['anomaly_score'] for r in all_records])

pct = 80 if len(scores) < 100 else (90 if len(scores) < 500 else 95)
thr = np.percentile(scores, pct)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(scores, bins=min(60, len(scores)//2 + 1), color='steelblue', alpha=0.8)
axes[0].axvline(thr, color='red', lw=2, label=f'{pct}th pct = {thr:.3f}')
axes[0].set(title='Anomaly score distribution',
            xlabel='Score (higher = more unusual)', ylabel='Count')
axes[0].legend()
axes[1].scatter(recon_errs, iso_score, s=2, alpha=0.4, c=combined, cmap='hot')
axes[1].set(title='Reconstruction error vs Isolation Forest',
            xlabel='Reconstruction error', ylabel='Isolation Forest score')
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Combined score')
plt.tight_layout()
plt.show()

n_cands = int((scores >= thr).sum())
print(f'\n{pct}th percentile threshold : {thr:.4f}')
print(f'Number of candidates      : {n_cands}')
print('\nScoring complete — run Cell 4 to review candidates')


## Cell 3b — Profile Visualization

Plots 8 low-anomaly (ordinary stellar) and 8 high-anomaly (candidate emission-line) spectral profiles side by side. Saves figure to `/content/lapaz/figures/sample_profiles.png`.


In [ ]:
# ── Figure: Sample spectral profiles (low vs high anomaly score) ─────────────

import random, os
import matplotlib.pyplot as plt

low_score  = sorted(all_records, key=lambda r: r['anomaly_score'])[:8]
high_score = sorted(all_records, key=lambda r: r['anomaly_score'], reverse=True)[:8]

fig, axes = plt.subplots(2, 8, figsize=(18, 4))

for ax, r in zip(axes[0], low_score):
    ax.plot(r['profile'], 'k-', lw=0.9)
    ax.set_title(f'sc={r["anomaly_score"]:.3f}', fontsize=6)
    ax.set_ylim(-3, 3)
    ax.axis('off')

for ax, r in zip(axes[1], high_score):
    ax.plot(r['profile'], 'r-', lw=0.9)
    ax.set_title(f'sc={r["anomaly_score"]:.3f}', fontsize=6)
    ax.set_ylim(-3, 3)
    ax.axis('off')

axes[0][0].set_ylabel('Low score\n(ordinary)', fontsize=7)
axes[1][0].set_ylabel('High score\n(anomalous)', fontsize=7)

plt.suptitle(
    r'Representative normalised 1D spectral profiles $\hat{p}(x)$ extracted from the La Paz plates',
    fontsize=9)
plt.tight_layout()

os.makedirs('/content/lapaz/figures', exist_ok=True)
plt.savefig('/content/lapaz/figures/sample_profiles.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved -> /content/lapaz/figures/sample_profiles.png')


## Cell 4 — Human Review Widget

Interactive ipywidgets interface showing postage-stamp thumbnails of top-scoring candidates.

- **Green border** = approved, **Red border** = rejected  
- Each stamp shows the plate cutout (grey) with the 1D spectral profile overlaid (yellow)  
- Adjust the score threshold slider and click **Apply threshold** to re-filter  
- Click **Save masks** when done — produces one binary PNG mask per plate  

**Approval criteria:**
- Compact, round morphology unlike surrounding elongated stellar streaks  
- Unusual 1D profile (spike, flat top, strong asymmetry)  
- Anomalously high surface brightness relative to neighbours  


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Approve / Reject widget
# ═══════════════════════════════════════════════════════════════════════════════

_img_cache = {}
def _load_img(plate_name):
    if plate_name not in _img_cache:
        p = PLATES_DIR / plate_name
        _img_cache[plate_name] = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    return _img_cache[plate_name]

STAMP_PX = 70
PAGE_SZ  = 30

class ReviewWidget:
    def __init__(self, records, masks_dir, init_thr):
        self.records   = records
        self.masks_dir = masks_dir
        self.thr       = init_thr
        self.cands     = []
        self.decisions = {}
        self.page      = 0

        self.thr_sl    = FloatSlider(
            value=init_thr,
            min=float(np.percentile(scores, 50)),
            max=float(scores.max()),
            step=0.002, readout_format='.3f',
            description='Min score:',
            layout=Layout(width='500px'))
        self.apply_btn = Button(description='Apply threshold',
                                button_style='primary',
                                layout=Layout(width='170px'))
        self.prev_btn  = Button(description='Prev page',  layout=Layout(width='120px'))
        self.next_btn  = Button(description='Next page',  layout=Layout(width='120px'))
        self.yes_btn   = Button(description='Approve page', layout=Layout(width='140px'))
        self.no_btn    = Button(description='Reject page',  layout=Layout(width='140px'))
        self.save_btn  = Button(description='Save masks',
                                button_style='success', layout=Layout(width='130px'))
        self.status    = HTML('<b>Adjust threshold -> Apply -> review -> Save</b>')
        self.out       = Output()

        self.apply_btn.on_click(self._apply)
        self.prev_btn.on_click(lambda _: self._go(-1))
        self.next_btn.on_click(lambda _: self._go(+1))
        self.yes_btn.on_click(lambda _: self._set_page(True))
        self.no_btn.on_click(lambda _: self._set_page(False))
        self.save_btn.on_click(self._save)

        display(VBox([
            HBox([self.thr_sl, self.apply_btn]),
            HBox([self.prev_btn, self.next_btn,
                  self.yes_btn, self.no_btn, self.save_btn]),
            self.status, self.out,
        ]))
        self._apply()

    def _n_pages(self):
        return max(1, (len(self.cands) + PAGE_SZ - 1) // PAGE_SZ)

    def _apply(self, _=None):
        self.thr   = self.thr_sl.value
        self.cands = [r for r in self.records if r['anomaly_score'] >= self.thr]
        for i in range(len(self.cands)):
            if i not in self.decisions:
                self.decisions[i] = True
        self.page = 0
        self._draw()

    def _go(self, delta):
        self.page = max(0, min(self.page + delta, self._n_pages() - 1))
        self._draw()

    def _set_page(self, val):
        start = self.page * PAGE_SZ
        for i in range(start, min(start + PAGE_SZ, len(self.cands))):
            self.decisions[i] = val
        self._draw()

    def _draw(self):
        with self.out:
            clear_output(wait=True)
            if not self.cands:
                print('No candidates at this threshold — lower the slider and click Apply.')
                return

            COLS  = 6
            start = self.page * PAGE_SZ
            batch = self.cands[start:start + PAGE_SZ]
            rows  = (len(batch) + COLS - 1) // COLS

            fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 2.7, rows * 3.1))
            if rows == 1: axes = [axes]
            flat = [ax for row in axes for ax in (row if COLS > 1 else [row])]

            for li, (ax, r) in enumerate(zip(flat, batch)):
                gi  = start + li
                img = _load_img(r['plate'])
                ok  = self.decisions.get(gi, True)
                if img is not None:
                    cx, cy = r['cx'], r['cy']
                    y1 = max(0, cy - STAMP_PX); y2 = min(img.shape[0], cy + STAMP_PX)
                    x1 = max(0, cx - STAMP_PX); x2 = min(img.shape[1], cx + STAMP_PX)
                    stamp = img[y1:y2, x1:x2]
                    lo = np.percentile(stamp, 2); hi = np.percentile(stamp, 98)
                    ax.imshow(stamp, cmap='gray', origin='upper', vmin=lo, vmax=hi)
                for sp in ax.spines.values():
                    sp.set_linewidth(3)
                    sp.set_color('#22cc44' if ok else '#cc2222')
                axin = ax.inset_axes([0.0, 0.0, 1.0, 0.22])
                axin.plot(r['profile'], 'y-', lw=0.8)
                axin.axis('off')
                ax.set_title(
                    f'sc={r["anomaly_score"]:.3f}  re={r["recon_err"]:.3f}\n'
                    f'a={r["aspect"]:.1f}  {r["plate"][:16]}',
                    fontsize=5.5)
                ax.axis('off')

            for ax in flat[len(batch):]:
                ax.axis('off')

            n_app = sum(self.decisions.get(i, True) for i in range(len(self.cands)))
            plt.suptitle(
                f'Page {self.page+1}/{self._n_pages()}  |  '
                f'{n_app}/{len(self.cands)} approved  |  threshold={self.thr:.3f}\n'
                f'Green=approved  Red=rejected  |  '
                f'sc=anomaly score  re=reconstruction error  a=aspect ratio',
                fontsize=8)
            plt.tight_layout()
            plt.show()

            btn_rows = []
            for i in range(0, len(batch), COLS):
                row_btns = []
                for li in range(i, min(i + COLS, len(batch))):
                    gi = start + li
                    ok = self.decisions.get(gi, True)
                    b  = Button(
                        description='OK' if ok else 'X',
                        button_style='success' if ok else 'danger',
                        layout=Layout(width='78px', height='26px'))
                    def _toggle(_, g=gi):
                        self.decisions[g] = not self.decisions.get(g, True)
                        self._draw()
                    b.on_click(_toggle)
                    row_btns.append(b)
                btn_rows.append(HBox(row_btns))
            display(VBox(btn_rows))

            self.status.value = (
                f'<b>Page {self.page+1}/{self._n_pages()}  |  '
                f'{n_app}/{len(self.cands)} approved  |  threshold = {self.thr:.3f}</b>')

    def _save(self, _=None):
        approved = [self.cands[i] for i in range(len(self.cands))
                    if self.decisions.get(i, True)]
        by_plate = {}
        for r in approved:
            by_plate.setdefault(r['plate'], []).append(r)

        n_saved = 0
        for plate_name, recs in by_plate.items():
            img = _load_img(plate_name)
            if img is None: continue
            h, w = img.shape
            mp   = self.masks_dir / (Path(plate_name).stem + '.png')
            mask = (cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
                    if mp.exists() else np.zeros((h, w), np.uint8))
            for r in recs:
                radius = max(12, int(np.sqrt(r['area'] / np.pi) * 1.8))
                cv2.circle(mask, (r['cx'], r['cy']), radius, 255, -1)
            cv2.imwrite(str(mp), mask)
            n_saved += len(recs)

        self.status.value = (
            f'<b style="color:green">{n_saved} emission objects saved across '
            f'{len(by_plate)} mask file(s) — run Cell 5 to train classifier</b>')
        print(f'Saved {n_saved} objects across {len(by_plate)} plates')

        # Save plate/mask example figures
        try:
            ref_plate = sorted(by_plate.keys())[0]
            img_raw   = _load_img(ref_plate)
            mp_ref    = self.masks_dir / (Path(ref_plate).stem + '.png')
            mask_img  = cv2.imread(str(mp_ref), cv2.IMREAD_GRAYSCALE)
            r0, r1, c0, c1 = 200, 800, 200, 900
            plate_crop = img_raw [r0:r1, c0:c1]
            mask_crop  = mask_img[r0:r1, c0:c1]
            os.makedirs('/content/lapaz/figures', exist_ok=True)
            cv2.imwrite('/content/lapaz/figures/plate_example.png', plate_crop)
            cv2.imwrite('/content/lapaz/figures/mask_example.png',  mask_crop)
            print('Example plate/mask figures saved -> /content/lapaz/figures/')
        except Exception as e:
            print(f'Figure save skipped: {e}')


print('HOW TO USE:')
print('  1. Adjust the Min score slider — start at the default (high)')
print('  2. Click Apply -> postage stamps appear, sorted best->worst')
print('  3. Each stamp shows: the spectrum image + 1D profile (yellow)')
print('  4. Click OK/X under each stamp to approve or reject')
print('  5. Use Next/Prev to page through candidates')
print('  6. Click Save masks when done')
print()
print('WHAT TO APPROVE:')
print('  [+] Compact round-ish spot unlike surrounding elongated streaks')
print('  [+] Unusual 1D profile (spike, flat top, asymmetric)')
print('  [+] Very bright relative to neighbours of similar streak length')
print()
print('WHAT TO REJECT:')
print('  [-] Plate defects (scratches, dust, vignetting artefacts)')
print('  [-] Blended / overlapping spectra')
print('  [-] Normal elongated stellar streak (just brighter than average)')
print()

widget = ReviewWidget(all_records, MASKS_DIR, init_thr=np.percentile(scores, pct))


## Cell 5 — ResNet18 Classifier + CSV Export

**Part A:** Extracts 64×64 cutouts centred on approved mask positions (positives) and random background positions (negatives, 3:1 ratio).

**Part B:** Augmented DataLoader (random flips, rotation 0–360°, brightness/contrast jitter ±0.3).

**Part C:** Fine-tunes ImageNet-pretrained ResNet18 (BCEWithLogitsLoss with class-frequency pos_weight, AdamW + OneCycleLR + early stopping, patience=15).

**Part D:** Re-scores all candidates from Cell 3. Final score = 0.5 × AE anomaly score + 0.5 × classifier probability.

**Part E:** Exports ranked `emission_catalogue.csv` and downloads `lapaz_results.zip` containing the catalogue, binary masks, and model checkpoints.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Stage 2: ResNet18 cutout classifier + download results
#
# Stage 1 (autoencoder, Cell 3) flagged anomalous candidates.
# Stage 2 trains a ResNet18 binary classifier on 64x64 cutouts:
#   positive = approved emission object, negative = background streak.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Restore session if kernel was restarted ───────────────────────────────────
import importlib, sys

_missing = [m for m in ['cv2', 'torch', 'albumentations', 'segmentation_models_pytorch']
            if m not in sys.modules]
if _missing:
    import subprocess
    subprocess.run(['pip', 'install', '-q',
                    'segmentation-models-pytorch', 'albumentations',
                    'scikit-learn', 'opencv-python-headless'], check=True)

import cv2, numpy as np, random, zipfile, os, csv
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
import matplotlib.pyplot as plt
from google.colab import files

WORK       = Path('/content/lapaz')
PLATES_DIR = WORK / 'plates'
MASKS_DIR  = WORK / 'masks'
CKPT_DIR   = WORK / 'checkpoints'
PLATE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AE_CKPT    = CKPT_DIR / 'autoencoder.pth'
SPEC_LEN   = 64

_img_cache = {}
def _load_img(plate_name):
    if plate_name not in _img_cache:
        _img_cache[plate_name] = cv2.imread(
            str(PLATES_DIR / plate_name), cv2.IMREAD_GRAYSCALE)
    return _img_cache[plate_name]

try:
    _ = all_records
    print(f'all_records in memory: {len(all_records):,} spectra')
except NameError:
    print('Warning: all_records not found — Part D (re-scoring) will be skipped.')
    all_records = []

print(f'Device : {DEVICE}')

# ── Check masks ───────────────────────────────────────────────────────────────
mask_files = sorted(MASKS_DIR.glob('*.png'))
print(f'Masks available : {len(mask_files)}')
if not mask_files:
    raise RuntimeError(
        'No masks found!\n'
        'Complete Cell 4 first: review candidates and click "Save masks".')

labeled_pairs = []
for mp in mask_files:
    for ext in PLATE_EXTS:
        ip = PLATES_DIR / (mp.stem + ext)
        if ip.exists():
            labeled_pairs.append((ip, mp))
            break

print(f'Labeled pairs   : {len(labeled_pairs)}')
if not labeled_pairs:
    raise RuntimeError('Mask filenames do not match plate filenames.')


# ═══════════════════════════════════════════════════════════════════════════════
# Part A — Build cutout dataset from masks
# ═══════════════════════════════════════════════════════════════════════════════

CUTOUT   = 64
HALF     = CUTOUT // 2
NEG_MULT = 3      # negatives per positive

def extract_cutouts(plate_path, mask_path):
    img  = cv2.imread(str(plate_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(mask_path),  cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        return []
    h, w = img.shape

    n, labels, stats, centroids = cv2.connectedComponentsWithStats(
        (mask > 127).astype(np.uint8), 8)

    cutouts      = []
    pos_centres  = []
    for i in range(1, n):
        cx = int(centroids[i][0])
        cy = int(centroids[i][1])
        if cx < HALF or cy < HALF or cx + HALF > w or cy + HALF > h:
            continue
        stamp = img[cy-HALF:cy+HALF, cx-HALF:cx+HALF].copy()
        cutouts.append((stamp, 1))
        pos_centres.append((cx, cy))

    n_neg    = len(pos_centres) * NEG_MULT
    attempts = 0
    while len([c for c in cutouts if c[1] == 0]) < n_neg and attempts < n_neg * 20:
        attempts += 1
        cx = random.randint(HALF, w - HALF - 1)
        cy = random.randint(HALF, h - HALF - 1)
        if mask[cy-HALF:cy+HALF, cx-HALF:cx+HALF].max() > 0:
            continue
        if any(abs(cx-px) < HALF and abs(cy-py) < HALF for px, py in pos_centres):
            continue
        stamp = img[cy-HALF:cy+HALF, cx-HALF:cx+HALF].copy()
        cutouts.append((stamp, 0))

    return cutouts


print('\nExtracting cutouts from labeled plates ...')
all_cutouts = []
for ip, mp in labeled_pairs:
    cuts = extract_cutouts(ip, mp)
    all_cutouts.extend(cuts)

n_pos_total = sum(1 for _, l in all_cutouts if l == 1)
n_neg_total = sum(1 for _, l in all_cutouts if l == 0)
print(f'\nTotal cutouts : {len(all_cutouts):,}')
print(f'  Positives   : {n_pos_total:,}')
print(f'  Negatives   : {n_neg_total:,}')

if n_pos_total < 10:
    raise RuntimeError(
        'Fewer than 10 positive cutouts found.\n'
        'Check that your masks contain white circles and plate names match.')

# Preview sample cutouts
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
pos_ex = [s for s, l in all_cutouts if l == 1][:8]
neg_ex = [s for s, l in all_cutouts if l == 0][:8]
for ax, stamp in zip(axes[0], pos_ex):
    lo, hi = np.percentile(stamp, [2, 98])
    ax.imshow(stamp, cmap='gray', vmin=lo, vmax=hi)
    ax.set_title('POS', fontsize=7); ax.axis('off')
for ax, stamp in zip(axes[1], neg_ex):
    lo, hi = np.percentile(stamp, [2, 98])
    ax.imshow(stamp, cmap='gray', vmin=lo, vmax=hi)
    ax.set_title('NEG', fontsize=7); ax.axis('off')
plt.suptitle('Sample cutouts — top: positives  |  bottom: negatives', fontsize=9)
plt.tight_layout(); plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# Part B — Dataset + transforms
# ═══════════════════════════════════════════════════════════════════════════════

class CutoutDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.tf    = transform

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        stamp, label = self.items[i]
        s = stamp.astype(np.float32)
        mu, sigma = s.mean(), s.std() + 1e-6
        s = np.clip((s - mu) / sigma, -3, 3) / 6.0 + 0.5
        img3 = np.stack([s, s, s], axis=2)
        return self.tf(img3), label


train_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

random.shuffle(all_cutouts)
nv       = max(1, len(all_cutouts) // 5)
tr_items = all_cutouts[nv:]
vl_items = all_cutouts[:nv]

BATCH    = 64
train_dl = DataLoader(CutoutDataset(tr_items, train_tf), batch_size=BATCH,
                      shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(CutoutDataset(vl_items, val_tf),   batch_size=BATCH,
                      shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain : {len(tr_items):,} cutouts — {len(train_dl)} batches/epoch')
print(f'Val   : {len(vl_items):,} cutouts — {len(val_dl)} batches/epoch')


# ═══════════════════════════════════════════════════════════════════════════════
# Part C — Train ResNet18
# ═══════════════════════════════════════════════════════════════════════════════

CLS_CKPT = CKPT_DIR / 'resnet18_best.pth'
CLS_LAST = CKPT_DIR / 'resnet18_last.pth'
EPOCHS   = 60
LR       = 1e-3
PATIENCE = 15

clf = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
clf.fc = nn.Linear(clf.fc.in_features, 1)
clf    = clf.to(DEVICE)

n_pos_tr = sum(1 for _, l in tr_items if l == 1)
n_neg_tr = sum(1 for _, l in tr_items if l == 0)
pos_w    = torch.tensor([n_neg_tr / max(n_pos_tr, 1)], dtype=torch.float32).to(DEVICE)
criterion_cls = nn.BCEWithLogitsLoss(pos_weight=pos_w)

optimizer_cls = optim.AdamW(clf.parameters(), lr=LR, weight_decay=1e-4)
scheduler_cls = optim.lr_scheduler.OneCycleLR(
    optimizer_cls, max_lr=LR, epochs=EPOCHS,
    steps_per_epoch=len(train_dl), pct_start=0.1)
scaler_cls = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

best_val_acc = 0.0
no_impr      = 0
start_ep     = 0
history_cls  = {'tl': [], 'vl': [], 'ta': [], 'va': []}

if CLS_LAST.exists():
    ck = torch.load(CLS_LAST, map_location=DEVICE)
    clf.load_state_dict(ck['model'])
    optimizer_cls.load_state_dict(ck['optimizer'])
    start_ep     = ck['epoch'] + 1
    best_val_acc = ck.get('best_acc', 0.0)
    print(f'Resumed ResNet18 from epoch {start_ep}, best_acc={best_val_acc:.4f}')

n_params = sum(p.numel() for p in clf.parameters() if p.requires_grad)
print(f'\nResNet18 — {n_params/1e6:.1f}M params  |  pos_weight={pos_w.item():.2f}')
print(f'\n{"Ep":>4} | {"TrLoss":>8} | {"TrAcc":>7} | {"VlLoss":>8} | {"VlAcc":>7}')
print('-' * 48)

for ep in range(start_ep, EPOCHS):
    # Train
    clf.train(); tl = ta = 0.0
    for imgs, labels in train_dl:
        imgs   = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer_cls.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = clf(imgs)
        loss = criterion_cls(logits.float(), labels)
        scaler_cls.scale(loss).backward()
        scaler_cls.unscale_(optimizer_cls)
        torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
        scaler_cls.step(optimizer_cls)
        scaler_cls.update()
        scheduler_cls.step()
        tl += loss.item()
        preds = (torch.sigmoid(logits.detach()) > 0.5).float()
        ta += (preds == labels).float().mean().item()
    tl /= len(train_dl); ta /= len(train_dl)

    # Validate
    clf.eval(); vl = va = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs   = imgs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                logits = clf(imgs)
            vl += criterion_cls(logits.float(), labels).item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            va += (preds == labels).float().mean().item()
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(labels.cpu().numpy().flatten())
    vl /= len(val_dl); va /= len(val_dl)

    arr_p   = np.array(all_preds); arr_l = np.array(all_labels)
    recall  = (arr_p[arr_l==1] == 1).mean() if (arr_l==1).any() else 0.0
    spec    = (arr_p[arr_l==0] == 0).mean() if (arr_l==0).any() else 0.0

    for k, v in zip(['tl', 'vl', 'ta', 'va'], [tl, vl, ta, va]):
        history_cls[k].append(v)

    lr_now = optimizer_cls.param_groups[0]['lr']
    print(f'{ep+1:>4} | {tl:>8.4f} | {ta:>6.1%} | {vl:>8.4f} | {va:>6.1%}  '
          f'(recall={recall:.0%} spec={spec:.0%})  lr={lr_now:.1e}')

    state = {'epoch': ep, 'model': clf.state_dict(),
             'optimizer': optimizer_cls.state_dict(), 'best_acc': best_val_acc}
    torch.save(state, CLS_LAST)

    if va > best_val_acc:
        best_val_acc = va; no_impr = 0
        torch.save(state, CLS_CKPT)
        print(f'  -> best model saved (acc={best_val_acc:.4f})')
    else:
        no_impr += 1
        if no_impr >= PATIENCE:
            print(f'Early stop at epoch {ep+1}'); break

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history_cls['tl'], label='Train'); a1.plot(history_cls['vl'], label='Val')
a1.set(title='Loss', xlabel='Epoch'); a1.legend(); a1.grid(True, alpha=0.3)
a2.plot(history_cls['ta'], label='Train'); a2.plot(history_cls['va'], label='Val')
a2.set(title='Accuracy', xlabel='Epoch'); a2.legend(); a2.grid(True, alpha=0.3)
plt.suptitle(f'ResNet18 training — best val accuracy = {best_val_acc:.4f}')
plt.tight_layout(); plt.show()
print(f'\nBest validation accuracy = {best_val_acc:.4f}')


# ═══════════════════════════════════════════════════════════════════════════════
# Part D — Re-score all candidates from Cell 3 with the trained classifier
# ═══════════════════════════════════════════════════════════════════════════════

if all_records:
    print('\nRe-scoring all candidates from Cell 3 ...')
    ck = torch.load(CLS_CKPT, map_location=DEVICE)
    clf.load_state_dict(ck['model'])
    clf.eval()

    def record_to_tensor(r, tf):
        img = _load_img(r['plate'])
        if img is None: return None
        cx, cy = r['cx'], r['cy']
        h, w   = img.shape
        if cx < HALF or cy < HALF or cx + HALF > w or cy + HALF > h:
            return None
        stamp = img[cy-HALF:cy+HALF, cx-HALF:cx+HALF].astype(np.float32)
        mu, sigma = stamp.mean(), stamp.std() + 1e-6
        s    = np.clip((stamp - mu) / sigma, -3, 3) / 6.0 + 0.5
        img3 = np.stack([s, s, s], axis=2)
        return tf(img3)

    cls_probs           = [0.0] * len(all_records)
    batch_tensors, batch_indices = [], []

    for i, r in enumerate(all_records):
        t = record_to_tensor(r, val_tf)
        if t is None:
            continue
        batch_tensors.append(t)
        batch_indices.append(i)
        if len(batch_tensors) == 256 or i == len(all_records) - 1:
            with torch.no_grad():
                xb = torch.stack(batch_tensors).to(DEVICE)
                with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                    logits = clf(xb)
                probs = torch.sigmoid(logits).cpu().numpy().flatten()
            for j, idx in enumerate(batch_indices):
                cls_probs[idx] = float(probs[j])
            batch_tensors, batch_indices = [], []

    for i, r in enumerate(all_records):
        r['cls_prob']    = cls_probs[i]
        r['final_score'] = 0.5 * r.get('anomaly_score', 0.0) + 0.5 * cls_probs[i]

    all_records.sort(key=lambda r: r['final_score'], reverse=True)

    probs = np.array([r['cls_prob'] for r in all_records])
    final = np.array([r['final_score'] for r in all_records])
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
    axes[0].hist(probs, bins=50, color='coral', alpha=0.8)
    axes[0].set(title='ResNet18 classifier probability',
                xlabel='P(emission object)', ylabel='Count')
    axes[1].hist(final, bins=50, color='steelblue', alpha=0.8)
    axes[1].set(title='Final combined score', xlabel='Score', ylabel='Count')
    plt.tight_layout(); plt.show()

    n_confirmed = int((probs >= 0.5).sum())
    print(f'Candidates confirmed by classifier (p>=0.5) : {n_confirmed:,}')
    print(f'Total candidates scored                     : {len(all_records):,}')
else:
    print('Skipping Part D — re-run Cells 1-3 to score all candidates.')


# ═══════════════════════════════════════════════════════════════════════════════
# Part E — Export CSV catalogue + download everything
# ═══════════════════════════════════════════════════════════════════════════════

csv_path   = WORK / 'emission_catalogue.csv'
fieldnames = ['rank', 'plate', 'cx', 'cy', 'area', 'aspect',
              'anomaly_score', 'cls_prob', 'final_score']

records_to_export = all_records if all_records else []

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for rank, r in enumerate(records_to_export, 1):
        writer.writerow({
            'rank'         : rank,
            'plate'        : r['plate'],
            'cx'           : r['cx'],
            'cy'           : r['cy'],
            'area'         : r['area'],
            'aspect'       : f"{r['aspect']:.2f}",
            'anomaly_score': f"{r.get('anomaly_score', 0):.4f}",
            'cls_prob'     : f"{r.get('cls_prob', 0):.4f}",
            'final_score'  : f"{r.get('final_score', 0):.4f}",
        })

print(f'\nCSV catalogue written: {csv_path}  ({len(records_to_export):,} rows)')

if records_to_export:
    print(f'\n{"Rank":>4} | {"Plate":30} | {"cx":>5} {"cy":>5} | '
          f'{"AnoScore":>8} | {"ClsProb":>7} | {"Final":>7}')
    print('-' * 80)
    for rank, r in enumerate(records_to_export[:20], 1):
        print(f'{rank:>4} | {r["plate"]:30} | {r["cx"]:>5} {r["cy"]:>5} | '
              f'{r.get("anomaly_score", 0):>8.4f} | '
              f'{r.get("cls_prob", 0):>7.4f} | '
              f'{r.get("final_score", 0):>7.4f}')

print('\nPacking results ...')
zip_path = Path('/content/lapaz_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for mp in MASKS_DIR.glob('*.png'):
        zf.write(mp, f'masks/{mp.name}')
    zf.write(csv_path, 'emission_catalogue.csv')
    if CLS_CKPT.exists():
        zf.write(CLS_CKPT, 'checkpoints/resnet18_best.pth')
    if AE_CKPT.exists():
        zf.write(AE_CKPT,  'checkpoints/autoencoder.pth')

print(f'Zip size : {zip_path.stat().st_size/1024:.0f} KB')
print('Downloading ...')
files.download(str(zip_path))
print()
print('lapaz_results.zip downloaded — contains:')
print('   emission_catalogue.csv   <- ranked list of all candidates')
print('   masks/                   <- binary PNG masks (one per plate)')
print('   checkpoints/             <- trained ResNet18 + autoencoder')
